# Train YOLO nhận diện biển báo VN trên Google Colab

Runtime → Change runtime type → GPU (A100 / L4 với Colab Pro). Notebook tự tải dataset từ Hugging Face, kết quả train lưu lên Google Drive.

In [ ]:
!nvidia-smi

In [ ]:
# Đổi URL thành repo GitHub của nhóm (repo private: dùng https://<token>@github.com/<user>/<repo>.git)
REPO_URL = 'https://github.com/HaiDang193/Traffic-sign-in-Viet-Nam.git'
!git clone -q $REPO_URL /content/project
%cd /content/project
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/vnsign'
!mkdir -p $DRIVE

In [ ]:
# Tải + gộp + chia dataset (khoảng 5-10 phút). Bị lỗi 429 / ngắt giữa chừng thì chạy lại cell này.
!python scripts/run_pipeline.py --skip-train

## Train
Kết quả lưu thẳng lên Drive (`--project`) để không mất khi Colab ngắt kết nối. Mỗi thí nghiệm đặt một `--name` riêng.

In [ ]:
NAME = 's_960'
!python scripts/train.py --model yolo11s.pt --name $NAME --imgsz 960 --epochs 100 --project $DRIVE/runs

In [ ]:
# Nếu Colab bị ngắt giữa chừng: train tiếp từ last.pt
# !python scripts/train.py --model $DRIVE/runs/$NAME/weights/last.pt --name $NAME --resume

In [ ]:
from IPython.display import Image, display
RUN = f'{DRIVE}/runs/{NAME}'
for f in ['results.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'val_batch0_pred.jpg']:
    try:
        display(Image(filename=f'{RUN}/{f}', width=900))
    except FileNotFoundError:
        print('missing', f)
!cat $RUN/test_summary.json

In [ ]:
# Phân tích lỗi + bảng so sánh các lần train (lưu lên Drive)
!python scripts/error_analysis.py --weights $RUN/weights/best.pt --imgsz 960 --out $DRIVE/reports/error_analysis/$NAME
!python scripts/compare_runs.py --runs $DRIVE/runs --out $DRIVE/reports